In [9]:
# Simplest LLM workflow

# LLM Q-A --> take the user query to the llm then get the structured output, then store the output in the workflow state 



# See how langchain and langgraph works hand in hand. State Attributes ===> question : str , answer : str 


from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import NotRequired, TypedDict
from dotenv import load_dotenv 


In [10]:
load_dotenv()

True

In [26]:
model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite"
)

In [12]:
# create state 

class LLMState(TypedDict):
    question: str 
    answer: NotRequired[str] 

    

In [18]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from the state 
    
    question = state['question']

    # form a prompt 

    prompt = f"Answer the following question {question}"

    # ask that question to the llm 

    answer = model.invoke(prompt).content

    # update the answer in the state 

    state['answer'] = answer

    return state

In [19]:
# Create Graph 

graph = StateGraph(LLMState)

# add nodes 

graph.add_node('llm_qa', llm_qa)

# add edges

graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile 

workflow = graph.compile()

In [25]:
# execute 

initial_state: LLMState = {'question': "How far is moon from the earth?"}

final_state = workflow.invoke(initial_state)

print(final_state)

{'question': 'How far is moon from the earth?', 'answer': [{'type': 'text', 'text': 'On average, the Moon is about **238,855 miles (384,400 kilometers)** away from Earth. \n\nHowever, because the Moon\'s orbit around Earth is elliptical (oval-shaped) rather than a perfect circle, this distance changes constantly. \n\n* **Closest point (Perigee):** About **225,623 miles (363,300 km)**. When a full moon occurs here, it is called a "Supermoon" because it looks larger and brighter.\n* **Farthest point (Apogee):** About **252,088 miles (405,500 km)**.\n\n### To help visualize this distance:\n* **The "30 Earths" rule:** You could fit approximately 30 Earth-sized planets side-by-side in the space between the Earth and the Moon.\n* **Travel time:** If you could drive a car to the Moon at 60 mph (96 km/h) without stopping, it would take you about **166 days** to get there. Apollo astronauts took about **3 days** to reach the Moon in a spacecraft.\n* **Speed of light:** Light (and radio signals)

In [24]:
print(final_state['answer'][0]['text'])

On average, the Moon is about **384,400 kilometers (238,855 miles)** away from Earth. 

However, because the Moon’s orbit around Earth is not a perfect circle but rather an oval (elliptical) shape, this distance changes constantly:

*   **At its closest (Perigee):** The Moon is about **363,300 kilometers (225,623 miles)** away. (This is when we often see a "Supermoon").
*   **At its farthest (Apogee):** The Moon is about **405,500 kilometers (251,966 miles)** away.

### To put this distance into perspective:
*   **The Planet Scale:** You could fit all seven other planets of our solar system (Mercury, Venus, Mars, Jupiter, Saturn, Uranus, and Neptune) side-by-side in the space between the Earth and the Moon, and still have some room left over.
*   **Speed of Light:** Light travels incredibly fast, but it still takes about **1.3 seconds** for light (or a radio signal) to travel from the Moon to the Earth.
*   **Travel Time:** If you could drive a car to the Moon at a constant speed of 10

In [29]:
model.invoke(input = "How far is moon from the earth?").content[0]['text']

'The average distance from the Earth to the Moon is about **384,400 kilometers** (or about **238,855 miles**). \n\nTo put that into perspective:\n* **Time:** It takes about 3 days for a spacecraft to travel from Earth to the Moon.\n* **Size:** You could fit all the planets in our solar system side-by-side in the space between the Earth and the Moon.\n\nBecause the Moon travels in an elliptical (oval-shaped) orbit rather than a perfect circle, the distance changes constantly:\n* **Closest point (Perigee):** About 363,300 km (225,600 miles)\n* **Farthest point (Apogee):** About 405,500 km (252,200 miles)'